# KPI ventes depuis PostgreSQL

Ce notebook lit directement les donnees depuis PostgreSQL via `vw_sales_analysis`, calcule les KPI principaux et produit des visualisations prêtes pour l'analyse.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'analysis' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.db import test_connection
from analysis.queries import load_kpi_summary, load_sales_analysis, load_validation_checks

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.options.display.float_format = '{:,.2f}'.format
sns.set_theme(style='whitegrid', palette='deep')

OUTPUT_DIR = PROJECT_ROOT / 'analysis_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')

## 1. Connexion PostgreSQL et sanity checks

In [ ]:
test_connection()
validation_checks = load_validation_checks()
display(validation_checks)
assert validation_checks['passed'].all(), 'Au moins un controle de coherence a echoue.'

## 2. Chargement du DataFrame d'analyse

In [ ]:
df = load_sales_analysis()
df['order_month'] = df['order_date'].dt.to_period('M').dt.to_timestamp()
df['shipping_delay_days'] = (df['ship_date'] - df['order_date']).dt.days
df['manager_name'] = df['manager_name'].fillna('Unassigned')

print(f'Rows loaded: {len(df):,}')
print(f'Columns loaded: {len(df.columns)}')
display(df.head())

## 3. KPI globaux

In [ ]:
kpi_summary = load_kpi_summary()
display(kpi_summary)

global_kpis = pd.DataFrame([
    {'metric': 'total_sales', 'value': float(df['sales'].sum())},
    {'metric': 'total_profit', 'value': float(df['profit'].sum())},
    {'metric': 'total_orders', 'value': int(df['order_id'].nunique())},
    {'metric': 'total_quantity', 'value': int(df['quantity'].sum())},
    {'metric': 'return_rate_pct', 'value': round(100 * df.loc[df['is_returned'], 'order_id'].nunique() / max(df['order_id'].nunique(), 1), 2)},
    {'metric': 'average_discount', 'value': float(df['discount'].mean())},
    {'metric': 'average_margin_pct', 'value': round(100 * df['profit'].sum() / max(df['sales'].sum(), 1), 2)},
    {'metric': 'average_shipping_delay_days', 'value': float(df['shipping_delay_days'].mean())},
])
display(global_kpis)

## 4. Analyses temporelles, geographiques, clients et produits

In [ ]:
monthly_performance = (
    df.groupby('order_month', as_index=False)
    .agg(
        sales=('sales', 'sum'),
        profit=('profit', 'sum'),
        orders=('order_id', 'nunique'),
        returned_orders=('is_returned', 'sum'),
        average_shipping_delay_days=('shipping_delay_days', 'mean'),
    )
    .sort_values('order_month')
)

top_months = monthly_performance.nlargest(5, 'sales')[['order_month', 'sales', 'profit', 'orders']]
bottom_months = monthly_performance.nsmallest(5, 'sales')[['order_month', 'sales', 'profit', 'orders']]

category_performance = (
    df.groupby('category_name', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), quantity=('quantity', 'sum'))
    .sort_values(['sales', 'profit'], ascending=[False, False])
)
sub_category_performance = (
    df.groupby(['category_name', 'sub_category_name'], as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), quantity=('quantity', 'sum'))
    .sort_values(['sales', 'profit'], ascending=[False, False])
)
product_performance = (
    df.groupby('product_name', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), quantity=('quantity', 'sum'))
    .sort_values(['sales', 'profit'], ascending=[False, False])
)
unprofitable_high_volume_products = product_performance.query('profit < 0').sort_values('quantity', ascending=False).head(10)

segment_performance = (
    df.groupby('segment', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('sales', ascending=False)
)
customer_performance = (
    df.groupby(['customer_id', 'customer_name', 'segment'], as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), orders=('order_id', 'nunique'))
    .sort_values(['sales', 'profit'], ascending=[False, False])
)

market_performance = (
    df.groupby('market_name', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('sales', ascending=False)
)
country_performance = (
    df.groupby('country_name', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('sales', ascending=False)
)
region_performance = (
    df.groupby('region_name', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('sales', ascending=False)
)
city_performance = (
    df.groupby('city_name', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('sales', ascending=False)
)
manager_performance = (
    df.groupby('manager_name', as_index=False)
    .agg(sales=('sales', 'sum'), profit=('profit', 'sum'), orders=('order_id', 'nunique'))
    .sort_values('sales', ascending=False)
)

display(top_months)
display(bottom_months)
display(category_performance.head(10))
display(sub_category_performance.head(10))
display(unprofitable_high_volume_products)
display(segment_performance)
display(customer_performance.head(10))
display(market_performance)
display(country_performance.head(10))
display(region_performance.head(10))
display(city_performance.head(10))
display(manager_performance)

## 5. Graphiques et export

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(18, 22))

sns.lineplot(data=monthly_performance, x='order_month', y='sales', marker='o', ax=axes[0, 0])
axes[0, 0].set_title('Chiffre d\'affaires mensuel')
axes[0, 0].tick_params(axis='x', rotation=45)

sns.lineplot(data=monthly_performance, x='order_month', y='profit', marker='o', color='#d95f02', ax=axes[0, 1])
axes[0, 1].set_title('Profit mensuel')
axes[0, 1].tick_params(axis='x', rotation=45)

sns.barplot(data=category_performance.head(10), x='sales', y='category_name', ax=axes[1, 0])
axes[1, 0].set_title('Top categories par CA')

sns.barplot(data=sub_category_performance.head(10), x='profit', y='sub_category_name', ax=axes[1, 1])
axes[1, 1].set_title('Top sous-categories par profit')

sns.barplot(data=segment_performance, x='sales', y='segment', ax=axes[2, 0])
axes[2, 0].set_title('Performance par segment client')

sns.barplot(data=market_performance.head(10), x='sales', y='market_name', ax=axes[2, 1])
axes[2, 1].set_title('Performance par marche')

sns.barplot(data=region_performance.head(10), x='sales', y='region_name', ax=axes[3, 0])
axes[3, 0].set_title('Top regions par CA')

sns.barplot(data=manager_performance.head(10), x='profit', y='manager_name', ax=axes[3, 1])
axes[3, 1].set_title('Performance par manager')

plt.tight_layout()
plt.show()

export_path = OUTPUT_DIR / 'kpi_summary.csv'
global_kpis.to_csv(export_path, index=False)
print(f'KPI summary exported to: {export_path}')